In [ ]:
# Lock NumPy to prevent 2.x upgrade
import sys
print("Step 1: Record Kaggle's default NumPy...")
import numpy as np
default_numpy = np.__version__
print(f"   Default: {default_numpy}")

print("\nStep 2: Install ultralytics...")
import subprocess
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print("\nStep 3: Lock NumPy...")
subprocess.run(["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"], check=True)

print("\nOK Installation complete")
import numpy as np
from ultralytics import YOLO
print(f"   NumPy: {np.__version__}")

In [ ]:
import json
import os
import sys
import time
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

# Environment setup
if os.path.exists("/kaggle/input/wound-seg-code-v1"):
    sys.path.insert(0, "/kaggle/input/wound-seg-code-v1")
    print("[INFO] Using Kaggle code dataset")
    from evaluation.yolo_utils import yolo_result_to_metrics_input
    from evaluation.metrics import calculate_segmentation_metrics
    from data.yolo_config import create_yolo_config
else:
    sys.path.insert(0, str(Path.cwd().parent))
    print("[INFO] Using local src")
    from src.evaluation.yolo_utils import yolo_result_to_metrics_input
    from src.evaluation.metrics import calculate_segmentation_metrics
    from src.data.yolo_config import create_yolo_config

print("OK Imports complete")

In [ ]:
# Configure paths
if os.path.exists("/kaggle/input"):
    # Use baseline-outputs for training (has pre-converted YOLO labels)
    BASE_INPUT_PATH = Path("/kaggle/input/baseline-outputs/data_wound_seg")
    # Use wound-segmentation-images for evaluation (has PNG masks)
    EVAL_INPUT_PATH = Path("/kaggle/input/wound-segmentation-images/data_wound_seg")
    OUTPUT_PATH = Path("/kaggle/working")
else:
    BASE_INPUT_PATH = Path.cwd().parent / "data" / "wound_segmentation"
    EVAL_INPUT_PATH = BASE_INPUT_PATH  # Same for local
    OUTPUT_PATH = Path.cwd().parent / "outputs" / "mini_ablations"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

TEST_IMAGES_DIR = EVAL_INPUT_PATH / "test_images"
TEST_MASKS_DIR = EVAL_INPUT_PATH / "test_masks"

print(f"Training data: {BASE_INPUT_PATH}")
print(f"Evaluation data: {EVAL_INPUT_PATH}")
print(f"Test images: {TEST_IMAGES_DIR}")
print(f"Test masks: {TEST_MASKS_DIR}")
print(f"Output: {OUTPUT_PATH}")

if not TEST_IMAGES_DIR.exists():
    raise FileNotFoundError(f"Test images not found: {TEST_IMAGES_DIR}")
if not TEST_MASKS_DIR.exists():
    raise FileNotFoundError(f"Test masks not found: {TEST_MASKS_DIR}")

print("OK Paths configured")

In [ ]:
# Create YOLO config
config_path = OUTPUT_PATH / "wound_dataset.yaml"
config_result = create_yolo_config(
    dataset_root=BASE_INPUT_PATH,
    output_path=config_path,
    train_dir="train_images",
    val_dir="test_images",
    test_dir="test_images"
)

if not config_result["success"]:
    raise RuntimeError(f"Config failed: {config_result['error']}")

print(f"OK YOLO config: {config_path}")

In [ ]:
# Mini-ablation configurations
MINI_ABLATIONS = {
  "1_no_mosaic": {
    "name": "No Mosaic",
    "hypothesis": "Mosaic provides critical spatial diversity",
    "expected": "70-75% Dice (significant drop)",
    "params": {
      "mosaic": 0.0
    }
  },
  "2_no_color": {
    "name": "No Color Augmentations",
    "hypothesis": "Color augmentations alter diagnostic features",
    "expected": "78-82% Dice",
    "params": {
      "hsv_h": 0.0,
      "hsv_s": 0.0,
      "hsv_v": 0.0,
      "erasing": 0.0,
      "auto_augment": None
    }
  },
  "3_no_scale": {
    "name": "No Scale",
    "hypothesis": "Scale redundant with mosaic",
    "expected": "78-82% Dice",
    "params": {
      "scale": 0.0
    }
  },
  "4_minimal": {
    "name": "Minimal (Mosaic+Flip Only)",
    "hypothesis": "Only essential augmentations needed",
    "expected": "75-80% Dice",
    "params": {
      "degrees": 0.0,
      "translate": 0.0,
      "scale": 0.0,
      "hsv_h": 0.0,
      "hsv_s": 0.0,
      "hsv_v": 0.0,
      "erasing": 0.0,
      "auto_augment": None
    }
  },
  "5_no_translate": {
    "name": "No Translate",
    "hypothesis": "Translate redundant with mosaic",
    "expected": "78-82% Dice",
    "params": {
      "translate": 0.0
    }
  },
  "6_clinical_conservative": {
    "name": "Clinical Conservative",
    "hypothesis": "Reduced safe augmentations match baseline",
    "expected": "78-82% Dice",
    "params": {
      "degrees": 0.0,
      "scale": 0.3,
      "hsv_h": 0.0,
      "hsv_s": 0.0,
      "hsv_v": 0.2,
      "erasing": 0.0,
      "auto_augment": None
    }
  },
  "7_no_randaugment": {
    "name": "No RandAugment",
    "hypothesis": "Unpredictable transforms hurt clinical color",
    "expected": "78-82% Dice (similar or better)",
    "params": {
      "auto_augment": None
    }
  }
}

BASELINE_PARAMS = {
  "degrees": 0.0,
  "translate": 0.1,
  "scale": 0.5,
  "shear": 0.0,
  "perspective": 0.0,
  "flipud": 0.0,
  "fliplr": 0.5,
  "mosaic": 1.0,
  "mixup": 0.0,
  "copy_paste": 0.0,
  "hsv_h": 0.015,
  "hsv_s": 0.7,
  "hsv_v": 0.4,
  "bgr": 0.0,
  "erasing": 0.4,
  "auto_augment": "randaugment",
  "close_mosaic": 10
}

# Training parameters
RANDOM_SEEDS = [42, 123, 456]
BATCH_SIZE = 32
LEARNING_RATE = 0.015
WORKERS = 8
AMP = False
IMAGE_SIZE = 512
EPOCHS = 100
PATIENCE = 10

print("=" * 80)
print("MINI-ABLATIONS PRODUCTION RUN")
print("=" * 80)
print(f"\nConfigurations: 7")
print(f"Seeds per config: 3")
print(f"Total training runs: 21")
print(f"\nEstimated time: ~7-8 hours total on P100")
print(f"  (~1 hour per config × 7 configs)")
print("=" * 80)

In [ ]:
# Storage for all results
all_results = {}

# Loop through each mini-ablation config
for ablation_id, config in MINI_ABLATIONS.items():
    print("\n" + "=" * 80)
    print(f"MINI-ABLATION: {config['name']} ({ablation_id})")
    print("=" * 80)
    print(f"Hypothesis: {config['hypothesis']}")
    print(f"Expected: {config['expected']}")
    print(f"Parameters: {config['params']}")

    ablation_start = time.time()
    ablation_results = {}

    # Train with 3 seeds for statistical robustness
    for seed in RANDOM_SEEDS:
        print(f"\n--- Seed {seed} ---")

        # Merge baseline params with ablation-specific overrides
        augmentation_params = BASELINE_PARAMS.copy()
        augmentation_params.update(config['params'])

        # Initialize model
        model = YOLO("yolov8n-seg.pt")

        # Train
        training_start = time.time()
        results = model.train(
            data=str(config_path),
            batch=BATCH_SIZE,
            lr0=LEARNING_RATE,
            workers=WORKERS,
            amp=AMP,
            epochs=EPOCHS,
            imgsz=IMAGE_SIZE,
            patience=PATIENCE,
            seed=seed,
            save=True,
            project=str(OUTPUT_PATH / "yolo_training" / ablation_id),
            name=f"seed_{seed}",
            exist_ok=True,
            verbose=False,
            **augmentation_params
        )

        training_duration = time.time() - training_start
        print(f"Training: {training_duration/60:.1f} min")

        # Load best model
        best_model_path = OUTPUT_PATH / "yolo_training" / ablation_id / f"seed_{seed}" / "weights" / "best.pt"
        best_model = YOLO(str(best_model_path))

        # Prepare test pairs
        test_image_paths = sorted(TEST_IMAGES_DIR.glob("*.png"))
        test_pairs = []
        for img_path in test_image_paths:
            mask_path = TEST_MASKS_DIR / img_path.name
            if mask_path.exists():
                test_pairs.append((str(img_path), str(mask_path)))

        print(f"Evaluating {len(test_pairs)} images...")

        # Evaluate
        seed_results = []
        for img_path_str, mask_path_str in test_pairs:
            image_id = Path(img_path_str).stem

            img_rgb = cv2.imread(img_path_str)
            img_rgb = cv2.cvtColor(img_rgb, cv2.COLOR_BGR2RGB)

            yolo_results = best_model.predict(img_rgb, verbose=False)

            conversion_result = yolo_result_to_metrics_input(
                yolo_results[0],
                mask_path_str,
                image_id
            )

            if not conversion_result["success"]:
                print(f"WARNING  Skip {image_id}: {conversion_result['error']}")
                continue

            data = conversion_result["data"]
            metrics_result = calculate_segmentation_metrics(
                data["pred_mask"],
                data["gt_mask"],
                image_id
            )

            seed_results.append(metrics_result)

        # Calculate seed statistics
        seed_dice_scores = [r["dice_score"] for r in seed_results]
        seed_dice_mean = np.mean(seed_dice_scores)

        ablation_results[f"seed_{seed}"] = {
            "dice_mean": float(seed_dice_mean),
            "n_images": len(seed_results),
            "training_time_seconds": training_duration
        }

        print(f"OK Seed {seed}: Dice = {seed_dice_mean:.4f}")

    # Calculate ablation overall statistics
    all_dice = [r["dice_mean"] for r in ablation_results.values()]
    ablation_dice_mean = np.mean(all_dice)
    ablation_dice_std = np.std(all_dice)

    ablation_duration = time.time() - ablation_start

    all_results[ablation_id] = {
        "name": config["name"],
        "hypothesis": config["hypothesis"],
        "expected": config["expected"],
        "params": config["params"],
        "dice_mean": float(ablation_dice_mean),
        "dice_std": float(ablation_dice_std),
        "seed_results": ablation_results,
        "total_time_seconds": ablation_duration
    }

    print(f"\nOK {config['name']}: Dice = {ablation_dice_mean:.4f} ± {ablation_dice_std:.4f}")
    print(f"   Duration: {ablation_duration/60:.1f} min")

print("\n" + "=" * 80)
print("OK ALL MINI-ABLATIONS COMPLETE")
print("=" * 80)

In [ ]:
# Summary table
print("\n" + "=" * 80)
print("MINI-ABLATIONS RESULTS SUMMARY")
print("=" * 80)
print(f"\nBaseline (full augmentation): 80.0% Dice\n")

for ablation_id, result in all_results.items():
    dice = result["dice_mean"]
    std = result["dice_std"]
    diff = dice - 0.8000
    print(f"{ablation_id}: {result['name']}")
    print(f"  Dice: {dice:.4f} ± {std:.4f}")
    print(f"  Change: {diff:+.4f} ({diff*100:+.2f}%)")
    print(f"  Expected: {result['expected']}")
    print()

# Save results
results_path = OUTPUT_PATH / "mini_ablations_results.json"
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print(f"OK Results saved: {results_path}")

In [ ]:
# Completion status
completion_status = {
    "notebook": "mini_ablations_production",
    "status": "complete",
    "n_configs": len(MINI_ABLATIONS),
    "n_seeds_per_config": len(RANDOM_SEEDS),
    "baseline_dice": 0.8000,
    "results_summary": {
        ablation_id: {
            "name": result["name"],
            "dice_mean": result["dice_mean"],
            "dice_improvement_vs_baseline": result["dice_mean"] - 0.8000
        }
        for ablation_id, result in all_results.items()
    }
}

status_path = OUTPUT_PATH / "completion_status.json"
with open(status_path, 'w') as f:
    json.dump(completion_status, f, indent=2)

print(f"\nOK Completion status: {status_path}")
print("\n" + "=" * 80)
print("OK MINI-ABLATIONS PRODUCTION COMPLETE")
print("=" * 80)